        # 01 貯蓄と複利による資産形成

        **F2：資産形成と早期リタイア（FIRE）のシミュレーション**

        - 目的：**貯蓄率**と**複利**によって、資産がどのように増えるかを理解する（RQ1・RQ2）
        - 計算：`financial_model.py`（Notebook ではグラフと考察を担当）

> 架空のデータによる**教育用シミュレーション**です。将来の投資収益の予測・保証や、投資・金融商品の推奨ではありません。金額は**万円**。税・社会保険料・インフレ・手数料・年金は考慮していません。

## 準備：実験環境（最初に必ず実行）

F2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                               # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f2-financial-independence"   # リポジトリの中の F2 フォルダ（ルートが F2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f2_root(p):
    return (Path(p) / "financial_model.py").exists() and (Path(p) / "fire_analysis.py").exists()


def find_f2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F2 のルート
        if is_f2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f2_upload")
    for hit in glob.glob("/content/f2_upload/**/financial_model.py", recursive=True):
        if is_f2_root(Path(hit).parent):
            return Path(hit).parent
    return None


F2_ROOT = find_f2_root()
if F2_ROOT is None:
    raise RuntimeError("F2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F2_ROOT)
if str(F2_ROOT) not in sys.path:
    sys.path.insert(0, str(F2_ROOT))
if not (F2_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。fire_analysis.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import financial_model as fm
import fire_analysis as fa
import retirement_model as rm
import optimization as op
import fire_charts as fc

fc.setup_style()                     # 文字を大きくするなら fc.setup_style(1.3)
scenarios, settings = fm.load_scenarios()
named = {s["name"]: s for s in scenarios}
END_ACC = settings["accumulation_end_age"]      # 資産形成の計算終了年齢（80）
END_RET = settings["retirement_end_age"]        # リタイア後の計算終了年齢（100）
W = settings["withdrawal_rate"]                 # 取り崩し率（4%、教育用の仮定）

SAVE_FIGURES = False                 # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", fc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: (("not reached" if k == "fire_age" else "-") if r.get(k) is None else f.format(r[k]))
             for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F2_ROOT:", F2_ROOT)
print("シナリオ:", [s["label"] for s in scenarios], f"/ 取り崩し率 {W:.0%} / 資産形成 {END_ACC} 歳まで / リタイア後 {END_RET} 歳まで")

## モデル

$$ \text{Savings} = \text{Income} \times \text{Savings Rate} $$
$$ \text{Assets}_{t+1} = \text{Assets}_t \times (1 + r) + \text{Savings}_t $$

- **年初**の資産に収益率 $r$ を適用し、**年末**に貯蓄を拠出する
- 開始年齢の年から、終了年齢の前の年まで計算する（22〜79 歳の年 → 80 歳時点の資産まで）
- 「a 歳時点の資産」＝ a 歳の年の**年初**の資産

## 1. 初期資産・所得・貯蓄率・収益率

In [ ]:
# TODO：値を変えて、下のセルをもう一度実行しよう
start_age      = 22      # 資産形成を始める年齢
initial_assets = 0       # 初期資産（万円）
annual_income  = 400     # 年間所得（万円）
savings_rate   = 0.20    # 貯蓄率（0.2 = 20%）
annual_return  = 0.03    # 年間収益率（教育用の仮定）

print("年間貯蓄額 =", fm.annual_savings(annual_income, savings_rate), "万円")
print("生活費（所得 − 貯蓄） =", fm.living_expenses(annual_income, savings_rate), "万円")

## 2. 年齢別資産（複利計算）

In [ ]:
rows = fm.simulate_accumulation(start_age, END_ACC, initial_assets, annual_income, savings_rate, annual_return)
table([r for r in rows if r["age"] in (22, 23, 24, 30, 40, 50, 60, 64, 79)],
      [("age", "age", "{}"), ("start_assets", "start assets", "{:,.0f}"),
       ("investment_return", "return", "{:,.1f}"), ("savings", "savings", "{:,.0f}"),
       ("end_assets", "end assets", "{:,.0f}")])
print(f"65 歳時点の資産：{fm.assets_at(rows, 65):,.0f} 万円")

In [ ]:
#@title グラフを表示（コードは隠しています）
ages, assets = fm.assets_by_age(rows)
show(fc.plot_assets([(f"savings {savings_rate:.0%}, return {annual_return:.0%}", ages, assets, fc.RAMP[2])],
                    "Assets by age"), "01_assets_by_age")

## 3. 貯蓄率別の資産推移（RQ1）

所得 400 万円を固定し、貯蓄率だけを 10%〜50% に変えます（収益率 3%）。

In [ ]:
#@title グラフを表示（コードは隠しています）
rates = op.savings_rate_candidates()            # [0.1, 0.2, 0.3, 0.4, 0.5]
series = []
for rate, color in zip(rates, fc.ramp(len(rates))):
    rws = fm.simulate_accumulation(start_age, END_ACC, initial_assets, annual_income, rate, annual_return)
    a, v = fm.assets_by_age(rws)
    series.append((f"savings {rate:.0%}", a, v, color))
show(fc.plot_assets(series, f"Assets by savings rate (income {annual_income}, return {annual_return:.0%})"),
     "01_by_savings_rate")
for label, a, v, _c in series:
    print(f"{label}: 65 歳 {v[65 - a[0]]:,.0f} 万円")

## 4. 収益率別の資産推移（RQ2）

貯蓄率 20% を固定し、収益率だけを 0%〜8% に変えます。**収益率は教育用の仮定で、将来の収益を予測するものではありません。**

In [ ]:
#@title グラフを表示（コードは隠しています）
returns = op.return_candidates()                # [0.0, 0.02, 0.04, 0.06, 0.08]
series = []
for ret, color in zip(returns, fc.ramp(len(returns))):
    rws = fm.simulate_accumulation(start_age, END_ACC, initial_assets, annual_income, savings_rate, ret)
    a, v = fm.assets_by_age(rws)
    series.append((f"return {ret:.0%}", a, v, color))
show(fc.plot_assets(series, f"Assets by annual return (savings rate {savings_rate:.0%})"), "01_by_return")

## 5. 複利の効果：貯めたお金と、増えたお金

65 歳までに**拠出した貯蓄の合計**（灰色）と、**投資収益の合計**（青）に分けます。

In [ ]:
#@title グラフを表示（コードは隠しています）
contrib, gains = [], []
for ret in returns:
    rws = fm.simulate_accumulation(start_age, 65, initial_assets, annual_income, savings_rate, ret)
    contrib.append(fm.total_contributions(rws) + initial_assets)
    gains.append(fm.total_investment_return(rws))
show(fc.plot_contributions_vs_gains([f"{r:.0%}" for r in returns], contrib, gains,
                                    "Assets at 65: contributions vs compound gains (by return)"),
     "01_contrib_vs_gains")

## アルゴリズムと計算量

| 処理 | 方法 | 計算量 |
| --- | --- | --- |
| 年齢別資産 | for ループで 1 年ずつ更新（複利） | $O(T)$、$T$ = 年数 |
| 貯蓄率・収益率を $K$ 通り試す | それぞれシミュレーション | $O(KT)$ |

複利は「前の年の結果を使って次の年を計算する」**逐次的な**計算です（Notebook ではループの結果をグラフにしているだけ）。

## 学生向け TODO

- [ ] **TODO 1**：「1. 初期資産・所得・貯蓄率・収益率」のセルで、貯蓄率を変更する
- [ ] **TODO 2**：収益率を変更する（負の収益率も試す）
- [ ] **TODO 3**：初期資産を変更する（例：500 万円）
- [ ] **TODO 4**：結果を比較する：貯蓄率を 10% 上げる効果と、収益率を 1% 上げる効果はどちらが大きいか（何歳時点で比べるかで変わるか）

## 考察（チームで記入）

- **TODO**：複利の効果が大きくなるのはどんなときか（期間・収益率・初期資産）
- **TODO**：このモデルで考えていないこと（税・手数料・インフレ・収益率の変動）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索・二分探索、第6章 挿入ソート）
- F2 リポジトリの `README.md`（モデルの仮定、4% ルールとその限界、計算量、モデルの限界と倫理的配慮）
- データ：`data/sample_scenarios.json`（この教材のための**架空のデータ**）